# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer Exploration with `mlcroissant`
This notebook demonstrates how to explore the FAIR² dataset, *Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution*, using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is described by a Croissant schema accessible at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

*Note: All dataset entities (record sets, fields, columns, etc.) are referenced only using their `@id` as per FAIR metadata conventions.*

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load the dataset's Croissant schema and metadata using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Let's count and inspect the record sets contained in the Croissant dataset, and display their available fields using only the `@id` to reference them.

In [ ]:
# List all record sets by @id and their fields' @id
print("Available record sets and their fields (@id):\n")
record_sets = list(dataset.record_sets)
for rs in record_sets:
    rs_metadata = dataset.record_set_metadata(rs)
    field_ids = [field['@id'] for field in rs_metadata['field']]
    print(f"- Record set @id: {rs}")
    print(f"  Fields @id: {field_ids}\n")

Let us view some example records from each record set—referenced by their `@id`.

In [ ]:
# Preview a few records from each record set using their @id
for rs in record_sets:
    print(f"Example records for record set @id: {rs}")
    for i, rec in enumerate(dataset.records(record_set=rs)):
        print(rec)
        if i >= 1:
            break
    print("-")

## 3. Data Extraction
Now, load the complete data for each record set into pandas DataFrames. All interactions reference record sets and fields by `@id` only.

In [ ]:
# Load data into DataFrames for all record sets by @id
dataframes = {}
for rs in record_sets:
    records = list(dataset.records(record_set=rs))
    df = pd.DataFrame(records)
    dataframes[rs] = df
    print(f"DataFrame columns for record set @id: {rs}: {df.columns.tolist()}")
# Show the head of the first available record set
if record_sets:
    first_rs = record_sets[0]
    print(f"\nPreview of first 5 records for record set @id: {first_rs}")
    display(dataframes[first_rs].head())

## 4. Exploratory Data Analysis (EDA)

Apply common exploration and processing steps. This demonstration uses only `@id` to reference fields and record sets. Adjust the `numeric_field_id` and `group_field_id` to those available in your dataset (see section 2 for IDs).

In [ ]:
# Specify the record set and field @ids for numeric analysis
# (Replace these with actual IDs from your dataset if needed)
example_rs = record_sets[0] if record_sets else None
example_df = dataframes[example_rs]

# Attempt to pick a numeric field (by heuristic: columns with float/integer type, or with 'age', 'interval', or similar in @id)
numeric_field_id = None
for col in example_df.columns:
    # Try to infer candidate numeric field (could be 'cr:age', 'cr:diagnosis_interval', etc.)
    if pd.api.types.is_numeric_dtype(example_df[col]) or ('age' in col.lower()) or ('interval' in col.lower()):
        numeric_field_id = col
        break

if numeric_field_id is None:
    raise Exception('Could not find a numeric field. Please update numeric_field_id manually!')

print(f"Using field @id for numeric EDA: {numeric_field_id}")

threshold = example_df[numeric_field_id].quantile(0.75)  # Filter above upper quartile as 'high' values
filtered_df = example_df[example_df[numeric_field_id] > threshold].copy()
print(f"Filtered records where {numeric_field_id} > {threshold:.2f} (upper quartile):")
print(filtered_df.head())

# Normalizing the numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try to find a group field (categorical, e.g., 'cr:sex' or similar)
group_field_id = None
for col in example_df.columns:
    if pd.api.types.is_object_dtype(example_df[col]) and col != numeric_field_id:
        group_field_id = col
        break
if group_field_id:
    print(f"\nGrouping by field @id: {group_field_id}")
    # Group mean of the numeric field by group field
    grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(grouped.head())

## 5. Visualization
Here, visualize the distribution of the selected numeric field, and (if grouping field is available) compare means across groups.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution of numeric field
plt.figure(figsize=(7, 4))
sns.histplot(example_df[numeric_field_id].dropna(), kde=True, bins=10)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

# Bar plot of mean by group if group_field_id is available
if group_field_id:
    plt.figure(figsize=(8, 4))
    sns.barplot(
        data=filtered_df, x=group_field_id, y=numeric_field_id, estimator='mean', ci=None
    )
    plt.title(f"Mean {numeric_field_id} by {group_field_id} (Filtered)")
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.xlabel(group_field_id)
    plt.show()

## 6. Conclusion
This notebook has demonstrated how to load and process the FAIR² colorectal cancer dataset using the `mlcroissant` library, referencing all entities by their `@id` per best practices. You can build upon this workflow to perform detailed analysis and modeling with full metadata provenance and transparency.